In [3]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [4]:
import pandas as pd

df = pd.read_csv(
    "IMDB Dataset.csv",
    encoding="utf-8",
    engine="python",
    on_bad_lines="skip"
)

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.columns

Index(['review', 'sentiment'], dtype='object')

In [6]:
from collections import Counter

def build_vocab(texts, vocab_size=10000):
    counter = Counter(" ".join(texts).split())
    vocab = {"<PAD>": 0, "<UNK>": 1}

    for i, (word, _) in enumerate(counter.most_common(vocab_size)):
        vocab[word] = i + 2

    return vocab

vocab = build_vocab(df["review"])
vocab_size = len(vocab)


In [7]:
import torch

def tokenize(text, vocab, max_len=128):
    tokens = [vocab.get(word, 1) for word in text.split()]
    tokens = tokens[:max_len]
    tokens += [0] * (max_len - len(tokens))
    return torch.tensor(tokens, dtype=torch.long)

In [8]:
from torch.utils.data import Dataset
import torch

class IMDBDataset(Dataset):
    def __init__(self, df, vocab, max_len=128):
        self.df = df
        self.vocab = vocab
        self.max_len = max_len
        self.sentiment_map = {'negative': 0, 'positive': 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
      text = self.df.iloc[idx]["review"]
      sentiment = self.df.iloc[idx]["sentiment"]
      label = 1 if sentiment == "positive" else 0
      tokens = [self.vocab.get(word, 1) for word in text.split()]
      tokens = tokens[:self.max_len]
      tokens += [0] * (self.max_len - len(tokens))
      input_ids = torch.tensor(tokens, dtype=torch.long)
      attention_mask = (input_ids != 0).long()
      return input_ids,attention_mask, torch.tensor(label, dtype=torch.long)

In [9]:
from torch.utils.data import DataLoader
dataset = IMDBDataset(df, vocab)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)


In [11]:
batch = next(iter(train_loader))
print(type(batch))
print(len(batch))

inputs, labels,attention_mask = batch
print(inputs.shape)          # [batch_size, seq_len]
print(attention_mask.shape)  # [batch_size, seq_len]
print(labels.shape)          # [batch_size]



<class 'list'>
3
torch.Size([32, 128])
torch.Size([32])
torch.Size([32, 128])


In [12]:
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=128):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ff(x))
        return x

class MiniTransformer(nn.Module):
    def __init__(self, vocab_size, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 128, padding_idx=0)
        self.pos = PositionalEncoding(128)

        self.layers = nn.ModuleList([
            TransformerBlock(128, 4, 256) for _ in range(2)
        ])

        self.classifier = nn.Linear(128, num_classes)

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        x = self.pos(x)

        for layer in self.layers:
            x = layer(x)

        x = x.mean(dim=1)
        return self.classifier(x)


In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = MiniTransformer(vocab_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    model.train()
    total_loss = 0

    for input_ids,attention_mask,labels in train_loader:
      input_ids = input_ids.to(device)
      attention_mask = attention_mask.to(device)
      labels = labels.to(device)
      outputs = model(input_ids) #attention_mask)

      optimizer.zero_grad()
      outputs = model(input_ids)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()
      total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader):.4f}")


Epoch 1, Loss: 0.6435
Epoch 2, Loss: 0.5156
Epoch 3, Loss: 0.4093
Epoch 4, Loss: 0.2943
Epoch 5, Loss: 0.1990
Epoch 6, Loss: 0.1223
Epoch 7, Loss: 0.0802
Epoch 8, Loss: 0.0584
Epoch 9, Loss: 0.0558
Epoch 10, Loss: 0.0412


In [14]:
model.eval()

MiniTransformer(
  (embedding): Embedding(10002, 128, padding_idx=0)
  (pos): PositionalEncoding()
  (layers): ModuleList(
    (0-1): 2 x TransformerBlock(
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
      )
      (ff): Sequential(
        (0): Linear(in_features=128, out_features=256, bias=True)
        (1): ReLU()
        (2): Linear(in_features=256, out_features=128, bias=True)
      )
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    )
  )
  (classifier): Linear(in_features=128, out_features=2, bias=True)
)

In [15]:
def encode_text(text, vocab, max_len=128):
    tokens = [vocab.get(word, 1) for word in text.lower().split()]
    tokens = tokens[:max_len]
    tokens += [0] * (max_len - len(tokens))
    return torch.tensor(tokens, dtype=torch.long)


In [16]:
import torch
import torch.nn.functional as F

def predict_sentiment(text, model, vocab, device):
    model.eval()

    input_ids = encode_text(text, vocab)
    input_ids = input_ids.unsqueeze(0).to(device)  # [1, seq_len]

    with torch.no_grad():
        outputs = model(input_ids)      # [1, num_classes]
        probs = F.softmax(outputs, dim=1)
        pred = torch.argmax(probs, dim=1).item()

    label_map = {0: "Negative", 1: "Positive"}

    return label_map[pred], probs.cpu().numpy()


In [17]:
text = input("Enter the text:")

prediction, probabilities = predict_sentiment(
    text,
    model,
    vocab,
    device
)

print("Input:", text)
print("Prediction:", prediction)
print("Probabilities:", probabilities)


Enter the text:it was an amazing movie
Input: it was an amazing movie
Prediction: Positive
Probabilities: [[3.8348582e-05 9.9996161e-01]]
